# Dynamics via Trotterization

Here, we show a simple example of how to simulate dynamics using Trotterization. We first need to create an electronic Hamiltonian from a molecular geometry. Here, we use $\mathrm{C_2H_6S_2}$ (dimethyl disulfide) as an example. 

In [1]:
import warnings
warnings.filterwarnings("ignore")

from rdkit import Chem
from rdkit.Chem import AllChem
import cupynumeric as np
import trovolut as trv

In [2]:
mol = Chem.AddHs(Chem.MolFromSmiles("CSSC"))
AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
AllChem.UFFOptimizeMolecule(mol)

conf = mol.GetConformer()
atoms = [a.GetSymbol() for a in mol.GetAtoms()]
xyz = (
    f"{len(atoms)}\ndmds generated by RDKit\n"
    + "\n".join(
        f"{s} {conf.GetAtomPosition(i).x:.8f} "
        f"{conf.GetAtomPosition(i).y:.8f} "
        f"{conf.GetAtomPosition(i).z:.8f}"
        for i, s in enumerate(atoms)
    )
)

H_elec = trv.qm.ElectronicHamiltonian.from_xyz(
    xyz_source=xyz,
    basis="sto-3g",
    charge=0,
    spin_mult=1,
    use_gpu4pyscf=False,
    freeze_core=True,
    n_core_keep=2,
    n_virt_keep=1,
).to_qubits()

converged SCF energy = -864.654938672564


We then get the terms of the Hamiltonian and decompose it for later use in Trotterization.

In [3]:
qmeta = H_elec.decompose()
hamcoeffs = H_elec.terms()[1:] # ignore the identity term as e^{c*I} is just a global phase

print("first 5 terms:")
for term in qmeta["terms"][1:6]:
    print(term)

first 5 terms:
{'coeff': np.float64(8.681865873431676), 'pauli': 'Z0'}
{'coeff': np.float64(2.700276903751764e-08), 'pauli': 'Y0Z1Z2Z3Y4'}
{'coeff': np.float64(2.700276903751764e-08), 'pauli': 'X0Z1Z2Z3X4'}
{'coeff': np.float64(8.681865873431676), 'pauli': 'Z1'}
{'coeff': np.float64(2.700276903751764e-08), 'pauli': 'Y1Z2Z3Z4Y5'}


In [4]:
hams = [term['coef'] * term['matrix'] for term in hamcoeffs]

In [5]:
evol = np.linalg.expm(-1j * hams[0] * 0.01) 

psi0 = np.zeros(2**qmeta["n_qubits"], dtype=complex)
psi0[0] = 1.0  # initial state is |000...0>

dyn = evol @ psi0
dyn.shape

(64,)

In [6]:
psi0 = np.zeros(2**qmeta["n_qubits"], dtype=complex)
psi0[0] = 1.0  # initial state |000...0>
dyn = trv.qm.dynamics.product_formula(hams, t=1, r=10, order=2) @ psi0

In [7]:
dyn.shape

(64,)